# A geography chatbot with LangGraph

Run these cells from top to bottom in Google Colab. The bot answers geography questions and politely declines other topics. Type `stop` to end the conversation.

In [ ]:
%pip -q install langgraph langchain langchain-openai

In [ ]:
import os
from typing import Annotated, TypedDict

from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
model = init_chat_model("openai:gpt-6-luna")

SYSTEM = SystemMessage(
    content=(
        "You are a friendly geography tutor for students. Answer only questions "
        "about geography. If asked about another topic, politely say you can "
        "help only with geography and invite the student to ask a geography question."
    )
)

In [ ]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


def chatbot(state: State):
    reply = model.invoke([SYSTEM] + state["messages"])
    return {"messages": [reply]}


builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

In [ ]:
history = []

while True:
    text = input("You: ")
    if text.strip().lower() == "stop":
        print("Goodbye!")
        break

    result = graph.invoke({"messages": history + [HumanMessage(content=text)]})
    history = result["messages"]
    print("Bot:", history[-1].content)